# Interview Prep — Hard Questions


## P1: Metaclass Plugin Registry

In [ ]:
class PluginMeta(type):
    registry: dict = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # skip base class
            key = namespace.get('plugin_name', name.lower())
            mcs.registry[key] = cls
            print(f'Registered plugin: {key!r} -> {name}')
        return cls

class Plugin(metaclass=PluginMeta):
    def execute(self): raise NotImplementedError

class JSONPlugin(Plugin):
    plugin_name = 'json'
    def execute(self): return 'JSON processing'

class CSVPlugin(Plugin):
    plugin_name = 'csv'
    def execute(self): return 'CSV processing'

class XMLPlugin(Plugin):
    plugin_name = 'xml'
    def execute(self): return 'XML processing'

print('\nRegistry:', list(PluginMeta.registry.keys()))
for name, cls in PluginMeta.registry.items():
    print(f'  {name}: {cls().execute()}')

## P2: Sliding Window Rate Limiter

In [ ]:
import time
from collections import deque
import threading

class SlidingWindowRateLimiter:
    def __init__(self, max_requests: int, window_seconds: float):
        self.max_requests = max_requests
        self.window = window_seconds
        self._requests: dict[str, deque] = {}
        self._lock = threading.Lock()

    def is_allowed(self, client_id: str) -> bool:
        now = time.time()
        with self._lock:
            if client_id not in self._requests:
                self._requests[client_id] = deque()
            window = self._requests[client_id]
            while window and window[0] < now - self.window:
                window.popleft()
            if len(window) < self.max_requests:
                window.append(now)
                return True
            return False

limiter = SlidingWindowRateLimiter(max_requests=3, window_seconds=1.0)
results = [limiter.is_allowed('user1') for _ in range(5)]
print(f'Immediate 5 requests: {results}')  # [T,T,T,F,F]

time.sleep(1.1)  # wait for window to reset
results2 = [limiter.is_allowed('user1') for _ in range(3)]
print(f'After 1s reset: {results2}')  # [T,T,T]

## P3: Serialize/Deserialize Binary Tree

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Codec:
    def serialize(self, root) -> str:
        if not root:
            return 'null'
        result = []
        queue = deque([root])
        while queue:
            node = queue.popleft()
            if node:
                result.append(str(node.val))
                queue.append(node.left)
                queue.append(node.right)
            else:
                result.append('null')
        return ','.join(result)

    def deserialize(self, data: str):
        if data == 'null':
            return None
        vals = data.split(',')
        root = TreeNode(int(vals[0]))
        queue = deque([root])
        i = 1
        while queue and i < len(vals):
            node = queue.popleft()
            if i < len(vals) and vals[i] != 'null':
                node.left = TreeNode(int(vals[i]))
                queue.append(node.left)
            i += 1
            if i < len(vals) and vals[i] != 'null':
                node.right = TreeNode(int(vals[i]))
                queue.append(node.right)
            i += 1
        return root

codec = Codec()
root = TreeNode(1, TreeNode(2), TreeNode(3, TreeNode(4), TreeNode(5)))
s = codec.serialize(root)
print(f'Serialized: {s}')
restored = codec.deserialize(s)
print(f'Restored:   {codec.serialize(restored)}')
assert s == codec.serialize(restored)

## P4: Async Gather with Timeout

In [ ]:
import asyncio

async def fetch(url: str, delay: float) -> dict:
    await asyncio.sleep(delay)
    return {'url': url, 'status': 200}

async def main():
    # All complete within timeout
    results = await asyncio.gather(
        fetch('api/users', 0.1),
        fetch('api/posts', 0.2),
        fetch('api/comments', 0.15),
    )
    print('All results:', results)

    # With timeout — some may be cancelled
    tasks = [
        asyncio.create_task(fetch('api/fast', 0.05)),
        asyncio.create_task(fetch('api/slow', 2.0)),
    ]
    done, pending = await asyncio.wait(tasks, timeout=0.1)
    for t in pending:
        t.cancel()
    print(f'Done: {len(done)}, Cancelled: {len(pending)}')

asyncio.run(main())

## P5: Implement Observer with Weak References

In [ ]:
import weakref
from typing import Callable

class WeakEventBus:
    """Event bus that doesn't prevent GC of handlers."""

    def __init__(self):
        self._handlers: dict[str, list] = {}

    def subscribe(self, event: str, handler: Callable) -> None:
        if event not in self._handlers:
            self._handlers[event] = []
        if hasattr(handler, '__self__'):
            ref = weakref.WeakMethod(handler)
        else:
            ref = weakref.ref(handler)
        self._handlers[event].append(ref)

    def publish(self, event: str, *args, **kwargs) -> int:
        if event not in self._handlers:
            return 0
        alive = []
        called = 0
        for ref in self._handlers[event]:
            handler = ref()
            if handler is not None:
                handler(*args, **kwargs)
                alive.append(ref)
                called += 1
        self._handlers[event] = alive
        return called

bus = WeakEventBus()

class Logger:
    def __init__(self, name):
        self.name = name
    def log(self, msg):
        print(f'[{self.name}] {msg}')

l1 = Logger('Logger1')
l2 = Logger('Logger2')
bus.subscribe('event', l1.log)
bus.subscribe('event', l2.log)

print('Both alive:')
n = bus.publish('event', 'Hello!')
print(f'  Called {n} handlers')

del l1  # garbage collected
print('\nAfter del l1:')
n = bus.publish('event', 'World!')
print(f'  Called {n} handlers')

## P6: Implement a Generic Result Type

In [ ]:
from typing import TypeVar, Generic, Callable, Optional
from dataclasses import dataclass

T = TypeVar('T')
U = TypeVar('U')

@dataclass
class Ok(Generic[T]):
    value: T
    def is_ok(self): return True
    def is_err(self): return False
    def map(self, f: Callable[[T], U]) -> 'Ok[U]': return Ok(f(self.value))
    def flat_map(self, f): return f(self.value)
    def unwrap(self): return self.value
    def unwrap_or(self, default): return self.value

@dataclass
class Err(Generic[T]):
    error: str
    def is_ok(self): return False
    def is_err(self): return True
    def map(self, f): return self
    def flat_map(self, f): return self
    def unwrap(self): raise ValueError(f'Called unwrap on Err: {self.error}')
    def unwrap_or(self, default): return default

Result = Ok | Err

def safe_divide(a: float, b: float) -> Result:
    if b == 0:
        return Err('Division by zero')
    return Ok(a / b)

def safe_sqrt(x: float) -> Result:
    if x < 0:
        return Err(f'Cannot take sqrt of {x}')
    return Ok(x ** 0.5)

# Chain operations
result = safe_divide(100, 4).flat_map(safe_sqrt).map(lambda x: round(x, 4))
print(f'sqrt(100/4) = {result.unwrap()}')  # 5.0

result2 = safe_divide(100, 0).flat_map(safe_sqrt)
print(f'Error: {result2.error}')  # Division by zero
print(f'unwrap_or: {result2.unwrap_or(0.0)}')